### Analysis of the severe slugging data (3W dataset)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from gtda.pipeline import Pipeline
from gtda.time_series import Resampler
from gtda.diagrams import PersistenceEntropy, Scaler, HeatKernel, BettiCurve
import numpy as np
from gtda.time_series import SingleTakensEmbedding, takens_embedding_optimal_parameters, TakensEmbedding
from sklearn.decomposition import PCA
from gtda.plotting import plot_point_cloud
from gtda.homology import VietorisRipsPersistence
from gtda.metaestimators import CollectionTransformer
import plotly.express as px
import statistics as stats

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

from tda_slugging.utils import read_files, find_shortest_file, batch_analyzer, mutual_information

# Data paths (type-3 slugging is committed; other types need the full 3W dataset)
DATA_3W_ALL      = str(REPO_ROOT / 'data' / '3W' / 'ALL')
DATA_3W_REAL     = str(REPO_ROOT / 'data' / '3W' / 'REAL')
DATA_3W_SIM      = str(REPO_ROOT / 'data' / '3W' / 'SIMULATED')
DATA_WELL        = str(REPO_ROOT / 'data' / 'well')
OUTPUTS_DIR      = str(REPO_ROOT / 'outputs')
IMAGES_DIR       = str(REPO_ROOT / 'images')
# Paths below require the full 3W dataset (download from https://github.com/ricardovvargas/3w_dataset):
# DATA_3W_NORMAL   = '/path/to/3w_dataset/data/0'
# DATA_3W_UNSTABLE = '/path/to/3w_dataset/data/4'


This notebook explores the severe slugging events part of the 3W dataset [R.E.V. Vargas et al. J. Pet. Sci. Eng.,
*181*, 106223 (2019)], partially simulated and and partially from offshore data. \
First we read the dataset and, from the multi-variate timeseries, we select only the wellhead pressure (P-TPT). Then we downsample the time series to 3000 points, in order to keep the computational effort for TDA to a manageable level. 

In [ ]:
find_shortest_file(DATA_3W_ALL, "P-TPT")

In [ ]:
slugging_signals, slugging_df = read_files(DATA_3W_ALL, "P-TPT",3000)

In [ ]:
PatoBar = 1/100000
slugging_df = slugging_df.apply(lambda x: x*PatoBar) 
slugging_signals = slugging_signals * PatoBar

We have a total of 105 severe slugging events: 33 real events and 75 simulated one (with OLGA)

In [ ]:
slugging_df

Let's show how one of these time series looks in detail 

In [ ]:
from random import gauss
from random import seed
from pandas.plotting import autocorrelation_plot
from statsmodels.graphics.gofplots import qqplot

fig_ss_analysis, data = plt.subplots(1, 3, figsize=(17, 3))
freq_units = 0.1/len(slugging_df[21])
print('freq units: %(units)4E Hz' %{"units":(freq_units)})

fig_ss_analysis.suptitle('slugging BHP stats')
data[0].set_xlabel('time')
data[0].set_ylabel('BHP (Bar)')
data[1].set_xlabel('freq (%(units).2E Hz)' %{"units":(freq_units)})
data[1].set_ylabel('power spectrum')
data[2].set_xlabel('BHP (Bar)')
data[2].set_ylabel('count')

data[1].set_xlim(2,1000)
data[1].set_yscale('log')
data[1].set_xscale('log')

#ss_BHP = slugging_df[35]
ss_BHP = slugging_df[23]

fft_ss = np.fft.rfft(ss_BHP)
fft_ss_abs = np.abs(fft_ss)
power_spectrum_ss = np.square(fft_ss_abs)


data[0].plot(ss_BHP)
data[1].plot(power_spectrum_ss)
ss_BHP.hist()

In [ ]:
# write file for making figures  
file = open("slugging_3W.dat", "w")
for i in ss_BHP:
    tmp = str(i) + "\n"
    file.writelines(tmp)
file.close()

We then perform Takens embedding [F. Takens, Lecture Notes in Mathematics *898* (1981)] to produce a point cloud form the time series.

In [ ]:
max_time_delay = 55
max_embedding_dimension = 12
stride = 3

signal = ss_BHP

optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
    signal, max_time_delay, max_embedding_dimension, stride=stride
    )
print('length of signal to analyze', int(len(signal)/stride))
print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")
embedding_dimension = optimal_time_delay
embedding_time_delay = optimal_time_delay

embedder = SingleTakensEmbedding(
    parameters_type="fixed", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_embedded = embedder.fit_transform(signal)

pca = PCA(n_components=3)
y_embedded_pca = pca.fit_transform(y_embedded)

style = {
            "x_range": [-15, 15],
            "y_range": [-15, 15],
            "z_range": [-5, 5],
            "marker": dict(size=12),
            "xaxis": dict(tickmode="linear", tick0=0, dtick=10),
            "yaxis": dict(tickmode="linear", tick0=-500, dtick=500),
        }

layout = {
        "xaxis": style["xaxis"],
        "yaxis": style["yaxis"],
        "xaxis_range": style["x_range"],
        "yaxis_range": style["y_range"]
    }

plotly_params_layout = {"layout": layout}

norm =  plot_point_cloud(y_embedded_pca, plotly_params=plotly_params_layout)
norm


In [ ]:
homology_dimensions = (0,1)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)
PerDiagram = VRP.fit_transform_plot(y_embedded[None,:,:])

In [ ]:
PE_BHP = PersistenceEntropy()
features = PE_BHP.fit_transform(PerDiagram)
features 

In [ ]:
PE_BHP = PersistenceEntropy(normalize=True)
features = PE_BHP.fit_transform(PerDiagram)
features 

In [ ]:
pers_H0 = []
pers_H1 = []
max_pers_H0 = []
max_pers_H1 = []

for PersDiag in PerDiagram: 
    for point in PersDiag:
        birth = point[0]
        death = point[1]
        dimension = point[2]
        persistence = abs(death - birth)
        pers_H0.extend([persistence] if dimension == 0 else [])
        pers_H1.extend([persistence] if dimension == 1 else [])
    max_pers_H0.append(np.amax(pers_H0))
    max_pers_H1.append(np.amax(pers_H1))

print('Maximum Persistence for BHP:')
print("Max Pers in homology dimension H_0", max_pers_H0 )
print("Max Pers in homology dimension H_1", max_pers_H1 )

In [ ]:
Betti = BettiCurve()
Betti_BHP = Betti.fit_transform_plot(PerDiagram)

print("mean of Betti_H0 for slug flow", stats.mean(Betti_BHP[0,0]))
print("mean of Betti_H1 for slug flow", stats.mean(Betti_BHP[0,1]))

In [ ]:
# write file for making figures  
file1 = open("slugW3_Pers_H0.dat", "w")
file2 = open("slugW3_Pers_H1.dat", "w")
for i in PerDiagram[0,:]:
    if i[2] == 0:
        tmp = str(i[0]) + str(' ') + str(i[1]) + "\n"
        file1.writelines(tmp)
    elif i[2] == 1:
        tmp = str(i[0]) + str(' ') + str(i[1]) + "\n"
        file2.writelines(tmp)
file1.close()
file2.close()

In [ ]:
batch_analyzer(slugging_df, 3, 20, 130)

In [ ]:
TE = TakensEmbedding(time_delay=125, dimension=7, stride=3)
homology_dimensions = (0, 1)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)
PE = PersistenceEntropy()
PE_norm = PersistenceEntropy(normalize=True)
Betti = BettiCurve()

slugging_point_cloud  = TE.fit_transform(slugging_signals)
slugging_diagrams = VRP.fit_transform(slugging_point_cloud)
slugging_entropy = PE.fit_transform(slugging_diagrams)
slugging_entropynorm = PE_norm.fit_transform(slugging_diagrams)
slugging_Betti = Betti.fit_transform(slugging_diagrams)

In [ ]:
tmp_entropies = slugging_entropy
Entropy_H0 = []
Entropy_H1 = []
tmp_normalized = slugging_entropynorm
Entropy_H0_norm = []
Entropy_H1_norm = []

for item in tmp_entropies:
    Entropy_H0.append(item[0])
    Entropy_H1.append(item[1])

for item in tmp_normalized:
    Entropy_H0_norm.append(item[0])
    Entropy_H1_norm.append(item[1])
    
Entropy_H1_series = pd.Series(Entropy_H1)
Entropy_H1_norm_series = pd.Series(Entropy_H1_norm)

entropies, ts = plt.subplots(2,figsize=(10,8),sharex = True)

entropies.suptitle('Slugging flow dataset persistent entropies')
ts[1].set_xlabel("# timeseries")
ts[0].set_ylabel('Entropy')
ts[1].set_ylabel('normalized  Entropy')

degrees = 70
plt.xticks(rotation=degrees)

ts[0].plot(Entropy_H1,'r-')
ts[0].plot(Entropy_H1_series.rolling(20).mean(), 'g-')
ts[0].plot(Entropy_H0,'b-')

ts[1].plot(Entropy_H1_norm,'r-')
ts[1].plot(Entropy_H1_norm_series.rolling(20).mean(), 'g-')
ts[1].plot(Entropy_H0_norm,'b-')

In [ ]:
stats_entropy, data = plt.subplots(1, 2,figsize=(10,4))
stats_entropy.suptitle('Normal operations dataset persistent entropies')

data[0].set_xlabel('Entropy H$_0$')
data[0].set_ylabel('count')
data[1].set_xlabel("Entropy H$_1$")
data[1].set_ylabel('count')

#data[0].set_xlim(8,10)
#data[1].set_xlim(0,8)
data[0].hist(Entropy_H0, bins=25)
data[1].hist(Entropy_H1, bins=25)

plt.tight_layout()

In [ ]:
stats_entropy, data = plt.subplots(1, 2,figsize=(10,4))
stats_entropy.suptitle('Normal operations dataset normalized persistent entropies')

data[0].set_xlabel('Normalized Entropy H$_0$')
data[0].set_ylabel('count')
data[1].set_xlabel("Normalized Entropy H$_1$")
data[1].set_ylabel('count')

data[0].hist(Entropy_H0_norm, bins=20)
data[1].hist(Entropy_H1_norm, bins=20)

plt.tight_layout()

In [ ]:
slugging_entropy.shape

In [ ]:
import statistics as stats
meanBettiH0 = []
medianBettiH0 = []
modeBettiH0 = []

for i in range(len(slugging_Betti[:,0,0])):
    meanBettiH0.append(stats.mean(slugging_Betti[i,0,:]))
    medianBettiH0.append(stats.median(slugging_Betti[i,0,:]))
    modeBettiH0.append(stats.mode(slugging_Betti[i,0,:]))
    
    
figBetti = px.line()
figBetti.add_scatter(y=(meanBettiH0), name="Mean of Betti curve H_0")
figBetti.add_scatter(y=(medianBettiH0), name="Median of Betti curve H_0")
figBetti.add_scatter(y=(modeBettiH0),name="Mode of Betti curve H_0")
figBetti.show()

In [ ]:
meanBettiH1 = []
medianBettiH1 = []
modeBettiH1 = []

for i in range(len(slugging_Betti[:,1,0])):
    meanBettiH1.append(stats.mean(slugging_Betti[i,1,:]))
    medianBettiH1.append(stats.median(slugging_Betti[i,1,:]))
    modeBettiH1.append(stats.mode(slugging_Betti[i,1,:]))
    
    
figBetti = px.line()
figBetti.add_scatter(y=(meanBettiH1), name="Mean of Betti curve H_1")
figBetti.add_scatter(y=(medianBettiH1), name="Median of Betti curve H_1")
figBetti.add_scatter(y=(modeBettiH1),name="Mode of Betti curve H_1")
figBetti.show()

In [ ]:
stats_entropy, data = plt.subplots(1, 2,figsize=(10,4))
stats_entropy.suptitle('Normal operations dataset - Means of Betti curves')

data[0].set_xlabel('Mean of Betti Curve for H$_0$')
data[0].set_ylabel('count')
data[1].set_xlabel("Mean of Betti Curve for H$_1$")
data[1].set_ylabel('count')

data[0].hist(meanBettiH0, bins=20)
data[1].hist(meanBettiH1)

plt.tight_layout()

In [ ]:
pers_H1 = []
pers_H0 = []
max_pers_H0 = []
max_pers_H1 = []

for PersDiag in slugging_diagrams: 
    for point in PersDiag:
        birth = point[0]
        death = point[1]
        dimension = point[2]
        persistence = abs(death - birth)
        pers_H0.extend([persistence] if dimension == 0 else [])
        pers_H1.extend([persistence] if dimension == 1 else [])
    max_pers_H0.append(np.amax(pers_H0))
    max_pers_H1.append(np.amax(pers_H1))
    pers_H0 = []
    pers_H1 = []
           
fig = px.line(title='Maximum Persistence for slugging BHP')
fig.add_scatter(y=max_pers_H1, name="Max Pers in homology dimension H_1")
fig.add_scatter(y=max_pers_H0, name="Max Pers in homology dimension H_0")

fig.show() 

In [ ]:
stats_entropy, data = plt.subplots(1, 2,figsize=(10,4))
stats_entropy.suptitle('Normal operations dataset - Max Persistence')

data[0].set_xlabel('Mean of Betti Curve for H$_0$')
data[0].set_ylabel('count')
data[1].set_xlabel("Mean of Betti Curve for H$_1$")
data[1].set_ylabel('count')

data[0].hist(max_pers_H0, bins = 25)
data[1].hist(max_pers_H1, bins = 25)

plt.tight_layout()

In [ ]:
VRP.plot(Xt=slugging_diagrams, sample=38)

In [ ]:
pca = PCA(n_components=3)
pca_slugging = pca.fit_transform(slugging_point_cloud[27])
plot_point_cloud(pca_slugging)

In [ ]:
PE(Xt=slugging_diagrams, sample=38)